<a href="https://colab.research.google.com/github/Parnika798/FinancialComplaintClassification/blob/main/Copy_of_Final_NLP_Project_Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

# Ensure the necessary libraries are installed
try:
    from transformers import (
        XLMRobertaTokenizer,
        AutoModelForSequenceClassification,
        TrainingArguments,
        Trainer,
        DataCollatorWithPadding # <-- IMPORT for dynamic padding
    )
    print("✅ Transformers library loaded successfully.")
except ImportError:
    print("❌ Transformers library not found. Run this in a new cell: !pip install transformers[torch] accelerate")
    exit()


✅ Transformers library loaded successfully.


In [ ]:
# ============================================================================
# REPRODUCIBILITY: SET RANDOM SEEDS (FINAL VERSION)
# ============================================================================

import random
import numpy as np
import torch
from transformers import set_seed as hf_set_seed

def set_seed(seed: int = 42):
    """
    Set all random seeds for full reproducibility across Python, NumPy,
    PyTorch, and HuggingFace Trainer.
    """
    # Python & NumPy
    random.seed(seed)
    np.random.seed(seed)

    # PyTorch (CPU & GPU)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # HuggingFace
    hf_set_seed(seed)

    # Ensure deterministic behavior (may slightly reduce performance)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
# ============================================================================
# STEP 1: LOAD AND PREPARE DATA (No Changes Here)
# ============================================================================
DATASET_PATH = "/content/sample_top5_translated.csv"
df = pd.read_csv(DATASET_PATH)
df = df.dropna(subset=['complaint_text', 'complaint_text_hindi', 'label'])
print(f"Dataset loaded with shape: {df.shape}")

df['bilingual_text'] = df['complaint_text'].fillna('') + " " + df['complaint_text_hindi'].fillna('')
label_encoder = LabelEncoder()
df['labels'] = label_encoder.fit_transform(df['label'])
num_labels = len(label_encoder.classes_)
print(f"Found {num_labels} classes: {label_encoder.classes_}")

X_train, X_test, y_train, y_test = train_test_split(
    df['bilingual_text'].tolist(),
    df['labels'].tolist(),
    test_size=0.2, random_state=42, stratify=df['labels']
)

Dataset loaded with shape: (25000, 3)
Found 5 classes: ['Checking or savings account' 'Credit card or prepaid card'
 'Credit reporting, credit repair services, or other personal consumer reports'
 'Debt collection' 'Mortgage']


In [ ]:
# ============================================================================
# STEP 2: TOKENIZE THE DATASET (FINAL - ALIGNED WITH XLM-R LARGE) 🧠
# ============================================================================

from transformers import AutoTokenizer, DataCollatorWithPadding
import torch

# Use LARGE model
model_name = "xlm-roberta-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("\nTokenizing the dataset (dynamic padding enabled)...")

# Increased max_length for better context utilization
train_encodings = tokenizer(
    X_train,
    truncation=True,
    padding=False,          # handled dynamically later
    max_length=256          # better for long complaints
)

test_encodings = tokenizer(
    X_test,
    truncation=True,
    padding=False,
    max_length=256
)

# Custom Dataset
class ComplaintDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)  # important
        return item

    def __len__(self):
        return len(self.labels)

# Create dataset objects
train_dataset = ComplaintDataset(train_encodings, y_train)
test_dataset = ComplaintDataset(test_encodings, y_test)

# Dynamic padding for efficiency
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


Tokenizing the dataset (dynamic padding enabled)...


In [ ]:
# ============================================================================
# STEP 3: MULTI-SEED TRAINING (COMPATIBLE + FINAL) 🚀
# ============================================================================

import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

seeds = [42, 123]
results = []

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc}

for s in seeds:
    print(f"\n🔁 Running training with seed = {s}")

    set_seed(s)

    # Load model
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels
    ).to(device)

    # OPTIONAL: Freeze lower layers
    for param in model.roberta.embeddings.parameters():
        param.requires_grad = False
    for layer in model.roberta.encoder.layer[:6]:
        for param in layer.parameters():
            param.requires_grad = False

    # Training arguments (FIXED for your version)
    training_args = TrainingArguments(
        output_dir=f'./results_seed_{s}',
        num_train_epochs=2,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        per_device_eval_batch_size=16,
        warmup_steps=300,
        weight_decay=0.01,
        logging_steps=200,
        eval_strategy="epoch",   # ✅ FIXED (not evaluation_strategy)
        save_strategy="no",
        fp16=True,
        seed=s,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    print("🚀 Training...")
    trainer.train()

    print("📊 Evaluating...")
    eval_results = trainer.evaluate()
    acc = eval_results['eval_accuracy']
    results.append(acc)

    print(f"✅ Seed {s} Accuracy: {acc:.4f}")

# ============================================================================
# FINAL RESULTS
# ============================================================================

mean_acc = np.mean(results)
std_acc = np.std(results)

print("\n" + "="*50)
print("=== FINAL MULTI-SEED RESULTS (XLM-R LARGE) ===")
print(f"Mean Accuracy: {mean_acc:.4f}")
print(f"Std Deviation: {std_acc:.4f}")
print(f"Formatted: {mean_acc*100:.2f} ± {std_acc*100:.2f}")
print("="*50)


🔁 Running training with seed = 42


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🚀 Training...


Epoch,Training Loss,Validation Loss,Accuracy
1,1.900318,0.435306,0.862800
2,1.325341,0.354713,0.884400


...

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🚀 Training...


Epoch,Training Loss,Validation Loss,Accuracy
1,1.998376,0.440801,0.859800
2,1.278330,0.352662,0.887000


📊 Evaluating...


✅ Seed 123 Accuracy: 0.8870

=== FINAL MULTI-SEED RESULTS (XLM-R LARGE) ===
Mean Accuracy: 0.8857
Std Deviation: 0.0013
Formatted: 88.57 ± 0.13


In [ ]:
# ============================================================================
# STEP 4: EVALUATE THE FINAL MODEL 🎯
# ============================================================================
print("\nEvaluating the final model...")
eval_results = trainer.evaluate()
final_accuracy = eval_results['eval_accuracy']

print("\n" + "="*35)
print("=== FINAL TRANSFORMER MODEL RESULTS ===")
print(f"✅ Final Accuracy: {final_accuracy:.4f} ({final_accuracy*100:.2f}%)")
print("="*35)

if final_accuracy >= 0.90:
    print("\n🚀🎯 CONGRATULATIONS! You have successfully reached the 90% accuracy target! 🎯🚀")


Evaluating the final model...



=== FINAL TRANSFORMER MODEL RESULTS ===
✅ Final Accuracy: 0.8838 (88.38%)


In [ ]:
import pickle

# ============================================================================
# STEP 5: SAVE THE FINAL MODEL AND ARTIFACTS 💾
# ============================================================================

# Define a directory to save everything
output_dir = "./final_transformer_model"

print(f"\nSaving model to {output_dir}...")

# 1. Save the fine-tuned model and its configuration
# This is the standard way to save a model from the Trainer.
trainer.save_model(output_dir)
print("✅ Model saved successfully.")

# 2. Save the tokenizer
# It's crucial to save the tokenizer so that new text is processed identically.
tokenizer.save_pretrained(output_dir)
print("✅ Tokenizer saved successfully.")

# 3. Save the Label Encoder
# We use pickle for standard Python objects like the label encoder.
with open(f'{output_dir}/label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)
print("✅ Label encoder saved successfully.")

print(f"\nAll artifacts are saved in the '{output_dir}' directory and are ready for deployment!")


Saving model to ./final_transformer_model...
✅ Model saved successfully.
✅ Tokenizer saved successfully.
✅ Label encoder saved successfully.

All artifacts are saved in the './final_transformer_model' directory and are ready for deployment!


In [ ]:
# ============================================================================
# STEP 6: TEST THE TRAINED MODEL WITH NEW EXAMPLES
# ============================================================================
# We can directly use the 'trainer', 'tokenizer', and 'label_encoder' objects
# that are already in our notebook's memory.

print("\n" + "="*50)
print("🧪 Testing the Fine-Tuned Model on New Complaints")
print("="*50)

# Get the best model from the trainer (it's loaded automatically)
model = trainer.model
model.eval() # Set the model to evaluation mode

def predict_complaint(text):
    """
    Takes a raw complaint text and uses the trained model in memory to
    predict its category.
    """
    # 1. Tokenize the input text
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256)

    # Move tensors to the same device as the model (e.g., GPU)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # 2. Perform inference
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    # 3. Convert logits to probabilities
    probabilities = torch.nn.functional.softmax(logits, dim=-1)[0]

    # 4. Get the predicted class index and confidence
    predicted_class_idx = torch.argmax(probabilities).item()
    confidence = probabilities[predicted_class_idx].item()

    # 5. Decode the prediction into a category name
    predicted_label = label_encoder.inverse_transform([predicted_class_idx])[0]

    return {
        "category": predicted_label,
        "confidence": confidence
    }

# --- Let's test it! ---
test_complaints = [
    "My credit card was charged twice for the same transaction at the restaurant.", # English
    "मेरे खाते से बिना अनुमति के पैसे काट लिए गए।", # Hindi (Money was debited from my account without permission.)
    "I have been trying to get a loan modification for months without any response.", # English
    "यह कंपनी मुझे हर दिन फोन करके परेशान कर रही है।", # Hindi (This company is harassing me by calling every day.)
    "There is an incorrect entry on my credit report that is lowering my score." # English
]

# Loop through the test complaints and print the predictions
for i, complaint in enumerate(test_complaints):
    result = predict_complaint(complaint)
    print(f"\nComplaint #{i+1}: '{complaint[:70]}...'")
    print(f"    ➡️ Predicted Category: {result['category']}")
    print(f"    Confidence: {result['confidence']:.2%}")


🧪 Testing the Fine-Tuned Model on New Complaints

Complaint #1: 'My credit card was charged twice for the same transaction at the resta...'
    ➡️ Predicted Category: Credit card or prepaid card
    Confidence: 98.28%

Complaint #2: 'मेरे खाते से बिना अनुमति के पैसे काट लिए गए।...'
    ➡️ Predicted Category: Credit reporting, credit repair services, or other personal consumer reports
    Confidence: 39.31%

Complaint #3: 'I have been trying to get a loan modification for months without any r...'
    ➡️ Predicted Category: Mortgage
    Confidence: 99.40%

Complaint #4: 'यह कंपनी मुझे हर दिन फोन करके परेशान कर रही है।...'
    ➡️ Predicted Category: Debt collection
    Confidence: 77.33%

Complaint #5: 'There is an incorrect entry on my credit report that is lowering my sc...'
    ➡️ Predicted Category: Credit reporting, credit repair services, or other personal consumer reports
    Confidence: 98.39%
